# Review checks (Nov 7)

Checks asked for by the site review. Everything here is **descriptive and after the 2026 test**: nothing changes the frozen
models, their features or their settings.

1. **Data checks** on the raw Kaggle file: columns, missing values, duplicates, overs with more than six legal balls,
   player names, no-result matches, byes and leg-byes.
2. **Extras check:** the wide / no-ball split used by the features against the raw `extra_type` column, and free hits.
3. **Where the gain comes from, independent of order:** XGBoost refit on every combination of four kinds of information
   (16 fits, 12 of them new), giving Shapley shares, the reverse-order ladder, and XGBoost against a scorecard-only model.

Run order: R0 cells (copied from `post_freeze_checks.ipynb`), then R1 onward. A GPU runtime (T4) is needed for R4.
New files go to Drive `data/` and never overwrite an existing file (R4 resumes its own file if the runtime drops).

## R0. Setup (copied unchanged from post_freeze_checks.ipynb)

In [ ]:
# ===== R0a. = T0 Setup and load (COPY-PASTE) =====
import json, os, time, datetime, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/IPL PROJECT ALL DOCS/data/'

df = pd.read_parquet(DATA + 'ipl_features.parquet')
with open(DATA + 'feature_list.json') as f: meta = json.load(f)
with open(DATA + 'freeze.json') as f:       FZ = json.load(f)
with open(DATA + 'calibration.json') as f:  CALJ = json.load(f)

FEATURES, CAT_FEATURES, CLASSES = meta['features'], meta['categorical'], meta['classes']
HIST = sum(meta['hist_features'].values(), [])
NUM = [c for c in FEATURES if c not in CAT_FEATURES]
NUM_H = NUM + HIST                        # B2-hist numeric inputs (82)
FEAT_HIST = list(FEATURES) + HIST         # X-hist inputs (87)
K = len(CLASSES)

# legal balls of 2008-2026; rain-affected matches have split 'rain' and stay out
d = df[(df['y'] >= 0) & df['split'].isin(['train', 'val', 'test'])].copy()
d['y'] = d['y'].astype(int)
for c in CAT_FEATURES:
    if not isinstance(d[c].dtype, pd.CategoricalDtype):
        d[c] = d[c].astype('category')

TR    = (d['split'] == 'train').to_numpy()     # 2008-2024: used only to pick X-hist's tree count
VA    = (d['split'] == 'val').to_numpy()       # 2025
FIT   = TR | VA                                # 2008-2025: every final model is refit here
TE    = (d['split'] == 'test').to_numpy()      # 2026: scored once, in T7
DEATH = (d['match_phase'] == 'death').to_numpy()

In [ ]:
# ===== R0b. = A0. d with rain matches + extra masks (COPY-PASTE) =====
d = df[(df['y'] >= 0) & df['split'].isin(['train', 'val', 'test', 'rain'])].copy()
d['y'] = d['y'].astype(int)
for c in CAT_FEATURES:
    if not isinstance(d[c].dtype, pd.CategoricalDtype):
        d[c] = d[c].astype('category')

TR    = (d['split'] == 'train').to_numpy()
VA    = (d['split'] == 'val').to_numpy()
FIT   = TR | VA
TE    = (d['split'] == 'test').to_numpy()
RAIN  = (d['split'] == 'rain').to_numpy()
DEATH = (d['match_phase'] == 'death').to_numpy()
IT    = TR & (d['year'] <= 2023).to_numpy()      # inner train, as in X0
IV    = TR & (d['year'] == 2024).to_numpy()      # inner val (early stopping for 2025 fits)
MATCH = d['match_id'].to_numpy()

print(d['split'].value_counts().to_dict())       # expect train 244269, val 15809, test 16282, rain 8101
print('rain matches by year:', d.loc[RAIN].groupby('year')['match_id'].nunique().to_dict())

In [ ]:
# ===== R0c. = T2. Helpers (COPY-PASTE: copies of models.ipynb functions) =====
EPS = 1e-15
def per_ball_logloss(P, y):
    return -np.log(np.clip(P[np.arange(len(y)), y], EPS, 1))

def per_ball_brier(P, y):
    O = np.zeros_like(P); O[np.arange(len(y)), y] = 1
    return ((P - O) ** 2).sum(axis=1)

def slice_paired(la, lb, mask, match, n_boot=2000, z_bonf=3.254, seed=0):
    """Mean of (la - lb) over the slice, 95% match-bootstrap interval, SE, Bonferroni flag. lb = 0 gives one model's interval."""
    diff, m = (la - lb)[mask], match[mask]
    codes, _ = pd.factorize(m)
    s, c = np.bincount(codes, weights=diff), np.bincount(codes)
    idx = np.random.default_rng(seed).integers(0, len(s), size=(n_boot, len(s)))
    boot = s[idx].sum(axis=1) / c[idx].sum(axis=1)
    mean, se = diff.mean(), boot.std()
    low, high = np.percentile(boot, [2.5, 97.5])
    return mean, low, high, se, abs(mean) > z_bonf * se

# --- B0 / B1 ---
def fit_freq(y): return np.bincount(y, minlength=K) / len(y)

WKT_BAND = pd.cut(d['c_team_wickets'], bins=[-1, 2, 5, 7, 9], labels=['0-2', '3-5', '6-7', '8-9'])
d['cell'] = d['match_phase'].astype(str) + ' | ' + WKT_BAND.astype(str) + ' wkts | inn ' + d['innings'].astype(str)
ALL_CELLS = [f'{ph} | {w} wkts | inn {i}' for ph in ['powerplay', 'middle', 'death']
             for w in ['0-2', '3-5', '6-7', '8-9'] for i in [1, 2]]

def fit_b1(rows, k=100):
    counts = (pd.crosstab(d.loc[rows, 'cell'], d.loc[rows, 'y'])
                .reindex(index=ALL_CELLS, columns=range(K), fill_value=0))
    prior = fit_freq(d.loc[rows, 'y'].values)
    n = counts.sum(axis=1).values[:, None]
    probs = np.where(n + k > 0, (counts.values + k * prior) / np.maximum(n + k, 1), prior)
    return pd.DataFrame(probs, index=ALL_CELLS, columns=CLASSES), counts.sum(axis=1)

def predict_b1(table, rows): return table.loc[d.loc[rows, 'cell']].values

# --- B2-hist ---
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning

def missing_groups(rows, cols):
    miss = d.loc[rows, cols].isna()
    miss = miss.loc[:, miss.any()]
    groups = {}
    for c in miss.columns:
        groups.setdefault(miss[c].values.tobytes(), []).append(c)
    return list(groups.values())

def fit_prep(rows, num, cat):
    X = d.loc[rows, num]
    lo, hi = X.quantile(0.005), X.quantile(0.995)
    few = X.nunique() <= 20
    lo[few], hi[few] = -np.inf, np.inf
    Xc = X.clip(lo, hi, axis=1)
    return {'num': num, 'cat': cat, 'lo': lo, 'hi': hi, 'mean': Xc.mean(), 'std': Xc.std().replace(0, 1),
            'groups': missing_groups(rows, num)}

def transform(prep, rows):
    X = d.loc[rows, prep['num']]
    Z = ((X.clip(prep['lo'], prep['hi'], axis=1) - prep['mean']) / prep['std']).fillna(0)
    flags = pd.DataFrame({'miss_' + g[0]: X[g[0]].isna().astype(float) for g in prep['groups']}, index=X.index)
    onehot = pd.get_dummies(d.loc[rows, prep['cat']], dtype=float)
    return pd.concat([Z, flags, onehot], axis=1)

def fit_lr(X, y, C=1.0):
    m = LogisticRegression(C=C, max_iter=3000)
    with warnings.catch_warnings():
        warnings.simplefilter('error', ConvergenceWarning)
        m.fit(X, y)
    return m

def to_logits(P, eps=1e-12): return np.log(np.clip(P, eps, 1))
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True); E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)
def apply_vector(P, a, b): return softmax(to_logits(P) * a + b)

# --- X-hist ---
import xgboost as xgb
XP = FZ['final_models']['X-hist']['params']
BEST = dict(max_depth=XP['max_depth'], min_child_weight=XP['min_child_weight'])
BASE_PARAMS = dict(objective='multi:softprob', eval_metric='mlogloss', tree_method='hist',
                   device='cuda', enable_categorical=True,
                   learning_rate=XP['learning_rate'], subsample=XP['subsample'],
                   colsample_bytree=XP['colsample_bytree'], random_state=0)

def fit_xgb_es(feats, fit_rows, stop_rows, **params):
    m = xgb.XGBClassifier(**BASE_PARAMS, **params, n_estimators=5000, early_stopping_rounds=50)
    m.fit(d.loc[fit_rows, feats], d.loc[fit_rows, 'y'],
          eval_set=[(d.loc[stop_rows, feats], d.loc[stop_rows, 'y'])], verbose=200)
    return m, m.best_iteration + 1, m.best_score

def fit_xgb_fixed(feats, rows, n_trees, **params):
    m = xgb.XGBClassifier(**BASE_PARAMS, **params, n_estimators=n_trees)
    m.fit(d.loc[rows, feats], d.loc[rows, 'y'], verbose=False)
    return m

print('xgboost', xgb.__version__, '| frozen X-hist settings:', BEST, XP)   # expect depth 4, mcw 100, lr .05, sub/col .8

In [ ]:
# ===== R0d. = A1. Feature groups (COPY-PASTE) =====
INMATCH, taken, G = list(FEATURES), set(), {}
def take(rule):
    cols = [c for c in INMATCH if c not in taken and rule(c)]
    taken.update(cols)
    return cols

G['delivery']     = take(lambda c: c in ('is_free_hit', 'prev_ball_outcome'))
G['chase']        = take(lambda c: any(k in c for k in ('required', 'need', 'pressure', 'runs_target')))
G['partnership']  = take(lambda c: 'partnership' in c or 'non_striker' in c or 'nonstriker' in c
                                   or c.startswith('ns_') or 'left_right' in c)
G['striker']      = take(lambda c: c.startswith('batter_') and c != 'batter_hand')
G['bowler_spell'] = take(lambda c: (c.startswith('bowler_') and c != 'bowler_cat')
                                   or 'spell' in c or 'since_last_bowled' in c)
G['momentum']     = take(lambda c: any(k in c for k in ('last6', 'last12', 'since', 'this_over')))
SITUATION = [c for c in INMATCH if c not in taken]
for k, v in meta['hist_features'].items():
    G['hist_' + k] = list(v)

for g, cols in G.items():
    print(f'{g:22s} {len(cols):2d}  {cols}')
print(f'{"SITUATION (kept)":22s} {len(SITUATION):2d}  {SITUATION}')
print('total:', sum(map(len, G.values())) + len(SITUATION))   # expect 87

In [ ]:
# ===== R0e. = A2. Drop-one-group refit (COPY-PASTE) =====
SEASON = {
    2025: dict(es_fit=IT, es_stop=IV, fit=TR,  score=VA),   # as in X3 (340 trees for the full model)
    2026: dict(es_fit=TR, es_stop=VA, fit=FIT, score=TE),   # as in T5 (301 trees for the full model)
}

def refit_without(drop, year):
    s = SEASON[year]
    feats = [c for c in FEAT_HIST if c not in set(drop)]
    _, n_trees, _ = fit_xgb_es(feats, s['es_fit'], s['es_stop'], **BEST)   # tree count, frozen way
    m = fit_xgb_fixed(feats, s['fit'], n_trees, **BEST)                    # refit with that count
    return m, n_trees, m.predict_proba(d.loc[s['score'], feats])

In [ ]:
# ===== R0f. = P0. Phase 1 setup (COPY-PASTE) =====
# needs T0 -> A0 -> T2 -> A1 -> A2 in this runtime
from scipy.stats import norm
from scipy.optimize import minimize_scalar

B26 = pd.read_parquet(DATA + 'bundle_2026.parquet')
y26, mt26, d26 = d.loc[TE, 'y'].to_numpy(), MATCH[TE], DEATH[TE]
assert (B26['y'].to_numpy() == y26).all() and (B26['match_id'].to_numpy() == mt26).all(), 'bundle order differs'
assert (B26['death'].to_numpy() == d26).all(), 'death mask differs'
ALL26 = np.ones(len(y26), bool)
SETS = [('all', ALL26), ('death', d26)]

TAGS = {'X-hist': 'x', 'LR (B2-hist +V)': 'l', 'B1': 'b1', 'B0': 'b0'}
PB = {name: B26[[f'p{t}_{k}' for k in range(K)]].to_numpy() for name, t in TAGS.items()}   # frozen 2026 predictions

saved = np.load(DATA + 'ablation_preds.npz')
P25_full, P26_full = saved['2025_full'], saved['2026_full']        # A3 refits (frozen recipe)
print('frozen X-hist vs A3 refit, largest |p difference|:', f"{np.abs(PB['X-hist'] - P26_full).max():.1e}")
# expect tiny (A3 reproduced 301 trees and 1.4787); anything under ~1e-2 is GPU noise

# copies of the A6 helpers
def fit_temperature(P, y):
    """The single T that minimises the log-loss of softmax(log p / T)."""
    Z = to_logits(P)
    nll = lambda T: per_ball_logloss(softmax(Z / T), y).mean()
    return minimize_scalar(nll, bounds=(0.5, 3.0), method='bounded').x

def ece_class(P, y, k, mask, n_bins=10):
    """Ball-weighted |mean predicted - actual share| over 10 equal-count bins, in points."""
    p, hit = P[mask, k], (y[mask] == k).astype(float)
    bins = np.array_split(np.argsort(p, kind='stable'), n_bins)
    return 100 * sum(len(b) * abs(p[b].mean() - hit[b].mean()) for b in bins) / len(p)

for balls, m in SETS:
    print(f'{balls:5s} {int(m.sum()):5d} balls | ' +
          ' | '.join(f'{k} {per_ball_logloss(v, y26)[m].mean():.4f}' for k, v in PB.items()))
# expect all:   16282 balls | X-hist 1.4787 | LR 1.4813 | B1 1.5204 | B0 1.5461
# expect death:  3583 balls | X-hist 1.5680 | LR 1.5764 | B1 1.6017 | B0 1.6434

In [ ]:
# ===== R0g. Save helper (COPY-PASTE) =====
def save_new(df_out, name):
    """Writes a new CSV to Drive data/. Refuses to overwrite an existing file."""
    path = DATA + name
    if os.path.exists(path):
        print('not saved, already exists:', path)
        return
    df_out.to_csv(path, index=False)
    print('saved', path, df_out.shape)

## R1. Data checks on the raw file

Needs the raw Kaggle file **IPL.csv** (kaggle.com/datasets/chaitu20/ipl-dataset2008-2025). R1a tries to download it with
`kagglehub`; if that fails, upload `IPL.csv` with the Files panel on the left and run R1a again.

In [ ]:
# ===== R1a. Load the raw file (COPY-PASTE) =====
RAW = '/content/IPL.csv'
if not os.path.exists(RAW):
    try:
        import kagglehub, glob, shutil
        path = kagglehub.dataset_download('chaitu20/ipl-dataset2008-2025')
        found = glob.glob(path + '/**/IPL.csv', recursive=True)
        shutil.copy(found[0], RAW)
    except Exception as e:
        print('download failed, upload IPL.csv by hand:', e)
raw = pd.read_csv(RAW, low_memory=False)
print('1. raw shape:', raw.shape)                       # expect (295732, 64)
print('2. columns:', list(raw.columns))

In [ ]:
# ===== R1b. Quality checks (COPY-PASTE) =====
col = lambda *names: next((n for n in names if n in raw.columns), None)
OVER, BALL, INN = col('over'), col('ball'), col('innings')
VALID, FACED = col('valid_ball'), col('balls_faced', 'binary_ball_faced')
SEASON_C = col('season', 'year')
print('using columns:', dict(over=OVER, ball=BALL, innings=INN, valid=VALID, faced=FACED, season=SEASON_C))

# 1. Missing values: only columns with any, as a share of rows
miss = raw.isna().mean().mul(100).round(2)
print('\n1. columns with missing values (% of rows):')
print(miss[miss > 0].sort_values(ascending=False).to_string())

# 2. Duplicates
print('\n2. fully identical rows:', int(raw.duplicated().sum()))
key = [c for c in ['match_id', INN, OVER, BALL, 'batter', 'bowler', 'non_striker', 'runs_total', 'extra_type', 'player_out'] if c in raw.columns]
print('   rows repeating', key, ':', int(raw.duplicated(subset=key).sum()))

# 3. Super overs and legal balls per over
real = raw[raw[INN].isin([1, 2])] if INN else raw
print('\n3. super-over rows (innings > 2):', len(raw) - len(real))     # expect 179
per_over = real.groupby(['match_id', INN, OVER])[VALID].sum()
print('   overs with more than 6 legal balls:', int((per_over > 6).sum()), 'of', len(per_over))
print('   overs with fewer than 6 legal balls:', int((per_over < 6).sum()), '(innings ending mid-over, rain)')
if (per_over > 6).any():
    print(per_over[per_over > 6].head(10))

# 4. Player names that differ only by case, spaces or dots
def norm_name(s): return s.str.lower().str.replace(r'[\s\.\-\']', '', regex=True)
names = pd.Series(pd.unique(pd.concat([real[c] for c in ['batter', 'bowler', 'non_striker'] if c in real.columns]).dropna()))
clash = names.groupby(norm_name(names)).agg(list)
clash = clash[clash.map(len) > 1]
print('\n4. distinct player names:', len(names), '| groups that differ only by case/spaces/dots:', len(clash))
print(clash.head(20).to_string())

# 5. Results: no-result and abandoned matches
res = col('result_type', 'win_outcome', 'result')
if res:
    per_match = real.groupby('match_id')[res].first()
    print('\n5. match results (', res, '):')
    print(per_match.fillna('(blank)').value_counts().to_string())
meth = col('method')
if meth:
    print('   matches with a method (e.g. D/L):', int(real.groupby('match_id')[meth].first().notna().sum()))
print('   matches:', real['match_id'].nunique())                     # expect 1243

# 6. Byes and leg-byes on legal balls
if 'extra_type' in raw.columns:
    legal = real[real[VALID] == 1]
    death = legal[legal[OVER] >= 15]
    for nm, part in [('all legal balls', legal), ('death legal balls', death)]:
        et = part['extra_type'].fillna('none').value_counts()
        print(f'\n6. extra types on {nm} ({len(part)}):')
        print((et / len(part) * 100).round(2).to_string())

## R2. Extras: the wide / no-ball split and free hits

In [ ]:
# ===== R2. Inferred extras vs the raw extra_type (COPY-PASTE) =====
ex = real[real[VALID] == 0].copy()
ex['inferred'] = np.where(ex[FACED] == 1, 'no-ball', 'wide')
print('1. wides and no-balls inferred from balls_faced against the raw extra_type:')
print(pd.crosstab(ex['extra_type'].fillna('(blank)'), ex['inferred'], margins=True).to_string())

# free hits rebuilt on the raw file with the features' rule
r = real.sort_values(['match_id', INN, OVER, BALL], kind='stable').copy()
is_nb = ((r[VALID] == 0) & (r[FACED] == 1)).astype(int)
legal_before = r.groupby(['match_id', INN])[VALID].cumsum() - r[VALID]
nb_before = is_nb.groupby([r['match_id'], r[INN], legal_before]).cumsum() - is_nb
r['free_hit'] = (nb_before > 0).astype(int)
fh = r[(r['free_hit'] == 1) & (r[VALID] == 1)]
print('\n2. legal free hits:', len(fh))
wk = col('wicket_kind')
if wk:
    yr = pd.to_datetime(fh['date']).dt.year if 'date' in fh.columns else fh[SEASON_C]
    out = fh[fh[wk].notna()]
    print('   dismissals on free hits by kind:')
    print(out.groupby([np.where(yr.loc[out.index] >= 2016, '2016 on', 'before 2016'), out[wk]]).size().to_string())

## R3. Where the gain comes from, in any order

Four kinds of information, as in the ladder: **S** the finer scorecard (17 columns), **T** player type (3), **C** in-match
context (39) and **H** history (28). XGBoost is refit, by the frozen 2026 recipe, on every one of the 16 combinations.
Without S, the model still gets B1's three columns (innings, phase and the wickets band), so the empty combination is
"XGBoost on what B1 knows" and every share is a share of the gain over that.

Four of the 16 fits already exist (the frozen model and the three ladder rungs) and are loaded, not refit.

In [ ]:
# ===== R3. The 16 combinations (COPY-PASTE) =====
import itertools
CONTEXT = ['delivery', 'chase', 'partnership', 'striker', 'bowler_spell', 'momentum']
CONTEXT_COLS = sum((G[g] for g in CONTEXT), [])
PLAYER_TYPE = [c for c in SITUATION if c in ('batter_hand', 'bowler_cat', 'matchup_type')]
SCORECARD = [c for c in SITUATION if c not in PLAYER_TYPE]
HIST_COLS = sum((G[g] for g in G if g.startswith('hist_')), [])
PARTS = {'S': SCORECARD, 'T': PLAYER_TYPE, 'C': CONTEXT_COLS, 'H': HIST_COLS}
print({k: len(v) for k, v in PARTS.items()})                       # expect S 17, T 3, C 39, H 28

d['wkt_band'] = pd.Categorical(WKT_BAND.astype(str))
B1_COLS = ['innings', 'match_phase', 'wkt_band']

def cols_for(combo):
    keep = set().union(*[PARTS[p] for p in combo]) if combo else set()
    feats = [c for c in FEAT_HIST if c in keep]
    if 'S' not in combo:
        feats = B1_COLS + feats
    return feats

COMBOS = [''.join(c) for r in range(5) for c in itertools.combinations('STCH', r)]
print(len(COMBOS), COMBOS)                                           # expect 16

lad = np.load(DATA + 'ladder_preds_2026.npz')
KNOWN = {'STCH': P26_full, 'S': lad['scorecard_state'], 'ST': lad['plus_player_type'], 'STC': lad['plus_in-match_context']}

## R4. Refit the 12 new combinations (GPU, about 20-30 min)

In [ ]:
# ===== R4. Refits, resumable (COPY-PASTE) =====
SHAP_FILE = DATA + 'shapley_preds_2026.npz'
P_COMBO = dict(np.load(SHAP_FILE)) if os.path.exists(SHAP_FILE) else {}
P_COMBO = {('' if k == 'none' else k): v for k, v in P_COMBO.items()}
TREES = {}
for combo in COMBOS:
    if combo in KNOWN or combo in P_COMBO:
        continue
    t0 = time.time()
    feats = cols_for(combo)
    s = SEASON[2026]
    _, n_trees, _ = fit_xgb_es(feats, s['es_fit'], s['es_stop'], **BEST)
    m = fit_xgb_fixed(feats, s['fit'], n_trees, **BEST)
    P_COMBO[combo] = m.predict_proba(d.loc[s['score'], feats])
    TREES[combo] = n_trees
    np.savez_compressed(SHAP_FILE, **{(k or 'none'): v for k, v in P_COMBO.items()})   # this notebook's own file
    print(f'{combo or "(B1 columns only)":18s} {len(feats):2d} features | {n_trees:4d} trees | {time.time() - t0:.0f} s')
P_ALL = {**P_COMBO, **KNOWN}
print('have', len(P_ALL), 'of 16')

## R5. Shapley shares, both ladders, and XGBoost against the scorecard-only model

In [ ]:
# ===== R5. Tables (COPY-PASTE) =====
from math import factorial
L = {c: per_ball_logloss(P_ALL[c], y26) for c in COMBOS}
n = 4
def weight(s): return factorial(s) * factorial(n - s - 1) / factorial(n)

rows = []
for balls, m in SETS:
    total = L['STCH'] - L['']
    tot_mu = total[m].mean()
    for p in 'STCH':
        phi = np.zeros_like(total)
        for c in COMBOS:
            if p in c:
                continue
            with_p = ''.join(x for x in 'STCH' if x in c + p)
            phi += weight(len(c)) * (L[with_p] - L[c])
        mu, lo, hi, _, bonf = slice_paired(phi, np.zeros_like(phi), m, mt26, z_bonf=norm.ppf(1 - 0.025 / 8))
        rows.append(dict(kind='shapley', balls=balls, part=p, value=mu, low=lo, high=hi, bonferroni=bonf, share=mu / tot_mu))
    for name, order in [('forward', 'STCH'), ('reverse', 'HCTS')]:
        prev = ''
        for p in order:
            cur = ''.join(x for x in 'STCH' if x in prev + p)
            mu, lo, hi, _, bonf = slice_paired(L[cur], L[prev], m, mt26, z_bonf=norm.ppf(1 - 0.025 / 8))
            rows.append(dict(kind=name, balls=balls, part=p, value=mu, low=lo, high=hi, bonferroni=bonf, share=mu / tot_mu))
            prev = cur
    # the B1+ comparison: final XGBoost minus the scorecard-only model, and the empty fit against B1 itself
    for kind, a, b in [('final_minus_scorecard_only', L['STCH'], L['S']),
                       ('b1_columns_fit_minus_B1', L[''], per_ball_logloss(PB['B1'], y26))]:
        mu, lo, hi, _, _ = slice_paired(a, b, m, mt26)
        rows.append(dict(kind=kind, balls=balls, part='', value=mu, low=lo, high=hi, bonferroni=np.nan, share=np.nan))
SH = pd.DataFrame(rows)
print(SH.round(4).to_string(index=False))
# checks: each 'share' column sums to 1.00 within shapley, forward and reverse, for each ball set
print(SH[SH['kind'].isin(['shapley', 'forward', 'reverse'])].groupby(['kind', 'balls'])['share'].sum().round(3))
print('per-combination death log-loss:', {c or 'B1 cols': round(L[c][d26].mean(), 4) for c in COMBOS})
save_new(SH, 'shapley_2026.csv')
save_new(pd.DataFrame([dict(combo=c or 'B1 cols', all=L[c].mean(), death=L[c][d26].mean()) for c in COMBOS]),
         'combo_logloss_2026.csv')